<a href="https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
!pip -q install duckdb huggingface_hub pyarrow

In [22]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [23]:
from huggingface_hub import login
import os

login(token=HF_TOKEN)

print("Hugging Face login successful")

Hugging Face login successful


In [24]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

print(dataset)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [25]:
# Peek first rows
rows = []

for i, row in enumerate(dataset):
    rows.append(row)
    if i == 4:
        break

rows

[{'report_date': datetime.date(2025, 1, 27),
  'client_hash_id': 'client_9958f0a7ae1df715',
  'content_hash_id': 'content_3b70a18ea133b2bb',
  'client_has_gsc': True,
  'client_has_ga4': True,
  'gsc_data_available': True,
  'ga4_data_available': False,
  'gsc_impressions': 30,
  'gsc_clicks': 0,
  'gsc_sum_position': 115,
  'gsc_avg_position': 3.8333333333333335,
  'ga4_pageviews': 0,
  'ga4_sessions': 0,
  'ga4_users': 0,
  'ga4_engaged_sessions': 0,
  'ga4_total_engagement_sec': 0,
  'sessions_organic': 0,
  'sessions_direct': 0,
  'sessions_referral': 0,
  'sessions_social': 0,
  'sessions_paid': 0,
  'sessions_ai': 0,
  'ai_chatgpt': 0,
  'ai_perplexity': 0,
  'ai_gemini': 0,
  'ai_copilot': 0,
  'ai_claude': 0,
  'ai_meta': 0,
  'ai_other': 0,
  'scroll_events': 0},
 {'report_date': datetime.date(2025, 1, 27),
  'client_hash_id': 'client_9958f0a7ae1df715',
  'content_hash_id': 'content_fe8e8155ce1d47a2',
  'client_has_gsc': True,
  'client_has_ga4': True,
  'gsc_data_available': 

In [26]:
from datetime import date

min_date = None
max_date = None
count = 0

for row in dataset:
    d = row["report_date"]

    if min_date is None or d < min_date:
        min_date = d

    if max_date is None or d > max_date:
        max_date = d

    count += 1

    # stop early for now
    if count == 100000:
        break

print("Rows checked:", count)
print("Min date:", min_date)
print("Max date:", max_date)

Rows checked: 100000
Min date: 2025-01-27
Max date: 2025-03-21


In [27]:
from collections import defaultdict

seen = set()
duplicates = []

for i, row in enumerate(dataset):
    key = (
        row["report_date"],
        row["client_hash_id"],
        row["content_hash_id"]
    )

    if key in seen:
        duplicates.append(key)
        break

    seen.add(key)

    if i == 100000:
        break

print("Rows checked:", len(seen))
print("Duplicate found:", len(duplicates) > 0)

if duplicates:
    print(duplicates[:5])

Rows checked: 100001
Duplicate found: False


In [28]:
columns_to_check = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "scroll_events"
]

missing = {col: 0 for col in columns_to_check}
total = 0

for row in dataset:
    total += 1

    for col in columns_to_check:
        if row[col] is None:
            missing[col] += 1

    if total == 100000:
        break

print("Rows checked:", total)

for col, count in missing.items():
    print(col, "missing:", count,
          "rate:", round(count/total*100, 2), "%")

Rows checked: 100000
gsc_impressions missing: 0 rate: 0.0 %
gsc_clicks missing: 0 rate: 0.0 %
gsc_avg_position missing: 1 rate: 0.0 %
ga4_pageviews missing: 0 rate: 0.0 %
ga4_sessions missing: 0 rate: 0.0 %
ga4_users missing: 0 rate: 0.0 %
scroll_events missing: 0 rate: 0.0 %


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [29]:
seen = set()
duplicate_found = False

for i, row in enumerate(dataset):
    key = (
        row["report_date"],
        row["client_hash_id"],
        row["content_hash_id"]
    )

    if key in seen:
        duplicate_found = True
        print("Duplicate found:", key)
        break

    seen.add(key)

    if i == 100000:
        break

print("Rows checked:", len(seen))
print("Duplicate found:", duplicate_found)

Rows checked: 100001
Duplicate found: False


One row represents the daily performance of one content item for one client.

The grain is:

`report_date × client_hash_id × content_hash_id`

The table contains daily observations from the warehouse performance table.

The observed sample checked:
- Minimum date: 2025-01-27
- Maximum date: 2025-03-21 (from the sampled rows)

The full warehouse release contains daily performance data from approximately 2025-01-27 to 2026-06-30.

Verification:
- Checked the grain using `(report_date, client_hash_id, content_hash_id)`.
- No duplicate combinations were found in 100,001 sampled rows.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

Features are fields available before making a prediction and can describe historical content performance.

Selected features:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- scroll_events


## Label

The label should represent the future outcome we want to predict.

Example:
- Future content performance change
- Future decline or improvement in engagement

The label must come from a future time window and should not be included as a feature.


## Context

These fields are used for grouping, joining, or analysis only:

- report_date
- client_hash_id
- content_hash_id

IDs are not model features because they only identify records and can cause memorization.


## Excluded

Excluded fields:

- Future performance metrics
- Any field calculated using the prediction outcome window

Reason:
These fields would not be available at prediction time and would introduce data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "scroll_events"
]

missing = {c: 0 for c in columns}
total = 0

for row in dataset:
    total += 1

    for c in columns:
        if row[c] is None:
            missing[c] += 1

    if total == 100000:
        break

print("Rows checked:", total)

for c in columns:
    print(
        c,
        "missing:",
        missing[c],
        "rate:",
        round(missing[c]/total*100, 2),
        "%"
    )


Rows checked: 100000
gsc_impressions missing: 0 rate: 0.0 %
gsc_clicks missing: 0 rate: 0.0 %
gsc_avg_position missing: 1 rate: 0.0 %
ga4_pageviews missing: 0 rate: 0.0 %
ga4_sessions missing: 0 rate: 0.0 %
ga4_users missing: 0 rate: 0.0 %
scroll_events missing: 0 rate: 0.0 %


The contract was verified using data checks.

Checks performed:

1. Grain verification:
- Checked duplicate combinations of:
  `report_date + client_hash_id + content_hash_id`
- No duplicate rows were found in 100,001 sampled records.


2. Date verification:
- Observed minimum date: 2025-01-27
- Observed maximum date in sample: 2025-03-21


3. Missing value verification:
Checked important performance fields:

- gsc_impressions: 0% missing
- gsc_clicks: 0% missing
- gsc_avg_position: approximately 0% missing
- ga4_pageviews: 0% missing
- ga4_sessions: 0% missing
- ga4_users: 0% missing
- scroll_events: 0% missing


The checks confirm that the observed table structure matches the expected daily content performance grain.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Data Limitations

This dataset has several limitations:

1. History availability differs between clients.
Some clients have shorter measurement histories, so comparisons across all dates may not always be fair.

2. GSC and GA4 availability are not identical.
Some rows may contain zero-filled analytics values when GA4 data is unavailable. These zeros should not always be interpreted as true absence of activity.

3. Future information leakage risk exists.
Features must only use information available before the prediction point. Future performance metrics cannot be used as model inputs.

4. Query-level context may contain overlapping time windows.
Window alignment must be checked before joining tables or creating features.

5. The data supports decision support and performance analysis, but it cannot explain all reasons behind content success or failure because external factors are not included.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.